In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os 
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By

from time import sleep

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ['Email'],os.environ['Password']

('dhruvilad1001@gmail.com', 'Littlepanda@10')

In [4]:
driver = webdriver.Chrome()

In [5]:
driver.get('https://www.linkedin.com/login')

In [6]:
driver.title

'LinkedIn Login, Sign in | LinkedIn'

In [6]:
email = driver.find_element(By.ID,'username')
email.send_keys(os.environ['Email'])

password = driver.find_element(By.ID,'password')
password.send_keys(os.environ['Password'])

password.submit()

In [7]:
url = "https://www.linkedin.com/in/savaliya-mukund27/"
# url = "https://www.linkedin.com/in/mitkumar-patel-031917266/"
# url = "https://www.linkedin.com/in/dhruvi-lad-7a900b352/"
driver.get(url)

In [8]:
profile_data = {}

In [9]:
page_source = driver.page_source
soup = BeautifulSoup(page_source,'lxml')

name = soup.find("h1")
if name:
    print(name.get_text().strip())
else:
    print("Name not found")

profile_data['name'] = name.get_text(strip=True) if name else None
profile_data['url'] = url


# name = soup.find('h1',{'class':'oHNNkEPkplriRaUkwcajehiFqypMDwTfPSgumYeQinline t-24 v-align-middle break-words'})
# name.get_text()

Mukund Savaliya


In [10]:
profile_data

{'name': 'Mukund Savaliya',
 'url': 'https://www.linkedin.com/in/savaliya-mukund27/'}

Experience

In [ ]:
page_source = driver.page_source
soup = BeautifulSoup(page_source,'lxml')

sections = soup.find_all('section',{'class':'artdeco-card pv-profile-card break-words mt2'})


In [ ]:
for sec in sections:
    if sec.find('div',{'id':'experience'}):
        experience = sec
# print(experience.get_text().strip())

In [ ]:
experience = experience.find_all('div', {'class':'ayGMfAhcBSxudvqbZimuHTADUEWsdaPUuw ANrTVEgBYLdYAAQEcIiKyNPMYMJCCSLkA axfzfYGBfFZOsNLwgUmKzZLCTXSZyuRXhTatFE'})     

In [ ]:
len(experience)

In [ ]:
exp = experience[0]

In [ ]:
def get_exp(exp):
    exp_dict = {}
    name = exp.find('div', {'class':'display-flex flex-wrap align-items-center full-height'})
    name = name.find('span', {'class':'visually-hidden'}).get_text().strip()


    try:
        duration = exp.find('span', {'class':'t-14 t-normal'})
        duration = duration.find('span', {'class':'visually-hidden'}).get_text().strip()
    except:
        # duration = exp.find('span', {'class':'t-14 t-normal'}) = None
        duration = None
        

    exp_dict['company_name'] = name
    exp_dict['duration'] = duration

    # check if company has multiple roles
    designations = exp.find_all('div', {'class':'lpNBjzeUzIYJHUgKfXFlOPssAwWhskOERCec'})
    item_list = []

    if designations and len(designations) > 0:
        # case 1: multiple roles
        for position in designations:
            spans = position.find_all('span', {'class':'visually-hidden'})
            item_dict = {}
            if len(spans) > 0: item_dict['designation'] = spans[0].get_text().strip()
            if len(spans) > 1: item_dict['role'] = spans[1].get_text().strip()
            if len(spans) > 2: item_dict['duration'] = spans[2].get_text().strip()
            if len(spans) > 3: item_dict['location'] = spans[3].get_text().strip()
            if len(spans) > 4: 
                item_dict['projects'] = spans[4].get_text().strip()
            else:
                item_dict['projects'] = ""
            item_list.append(item_dict)
    else:
        # case 2: single role company
        spans = exp.find_all('span', {'class':'visually-hidden'})
        if len(spans) >= 3:   # usually designation, duration, location exist
            item_dict = {}
            item_dict['designation'] = spans[0].get_text().strip()
            # item_dict['role'] = spans[1].get_text().strip() if len(spans) > 1 else ""
            item_dict['duration'] = spans[2].get_text().strip() if len(spans) > 2 else ""
            item_dict['location'] = spans[3].get_text().strip() if len(spans) > 3 else ""
            item_dict['projects'] = spans[4].get_text().strip() if len(spans) > 4 else ""
            item_list.append(item_dict)

    exp_dict['designations'] = item_list
    return exp_dict


# run for all experience blocks
item_list = []
for exp in experience:
    item_list.append(get_exp(exp))

item_list

try-1

In [ ]:
# --- imports ---
import json, re
# from bs4 import BeautifulSoup
# from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- helper: get experience tokens from the Experience section ---
def get_experience_tokens(driver, timeout=15):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.35);")

    try:
        exp_section = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.XPATH, "//section[.//span[text()='Experience']]")
            )
        )
    except:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, "//span[text()='Experience']"))
        )
        exp_section = driver.find_element(By.XPATH, "//span[text()='Experience']/ancestor::section[1]")

    exp_html = exp_section.get_attribute("outerHTML")
    exp_soup = BeautifulSoup(exp_html, "lxml")

    tokens = [s.get_text(strip=True) for s in exp_soup.select("span.visually-hidden") if s.get_text(strip=True)]
    return tokens

# --- helper: parse tokens into your target structure ---
def parse_experience_tokens(tokens):
    if tokens and tokens[0].strip().lower() == "experience":
        tokens = tokens[1:]

    roles_set = {
        "Full-time","Part-time","Internship","Contract","Self-employed",
        "Freelance","Apprenticeship","Volunteer","Temporary","Seasonal"
    }

    def is_duration(s: str) -> bool:
        s2 = s.lower()
        return (' to ' in s2 and ('present' in s2 or 'mo' in s2 or 'yr' in s2)) or (' yr' in s2) or (' mo' in s2)

    def is_location(s: str) -> bool:
        if is_duration(s): return False
        if s in roles_set: return False
        loc_keywords = ['india','remote','on-site','hybrid','onsite','work from home']
        if any(k in s.lower() for k in loc_keywords):
            return True
        return (',' in s and len(s.split()) >= 2)

    def is_meta(s: str) -> bool:
        return s in roles_set or is_duration(s) or is_location(s) or s.lower() in ('on-site','remote','hybrid')

    i, n = 0, len(tokens)
    result = []

    while i < n:
        lookahead = tokens[i:i+12]
        role_count = sum(1 for t in lookahead if t in roles_set)

        if role_count >= 2:
            company_name = tokens[i]; i += 1
            company_duration = None

            while i < n and (is_duration(tokens[i]) or tokens[i].lower() in ('on-site','remote','hybrid')):
                if is_duration(tokens[i]) and company_duration is None:
                    company_duration = tokens[i]
                i += 1

            entry = {'company_name': company_name, 'duration': company_duration, 'designations': []}

            while i < n:
                while i < n and is_meta(tokens[i]):
                    i += 1
                if i >= n: break

                designation = tokens[i]; i += 1
                role = tokens[i] if i < n and tokens[i] in roles_set else None
                if role is not None: i += 1
                dur = tokens[i] if i < n and is_duration(tokens[i]) else None
                if dur is not None: i += 1
                loc = tokens[i] if i < n and is_location(tokens[i]) else None
                if loc is not None: i += 1

                entry['designations'].append({
                    'designation': designation,
                    **({'role': role} if role else {}),
                    **({'duration': dur} if dur else {}),
                    **({'location': loc} if loc else {}),
                    'projects': ''
                })

                window = tokens[i:i+6]
                if sum(1 for t in window if t in roles_set) == 0 and any(is_duration(t) for t in window):
                    break

            result.append(entry)

        else:
            designation = tokens[i]; i += 1
            org = tokens[i] if i < n else None
            if org is not None: i += 1
            dur = tokens[i] if i < n and is_duration(tokens[i]) else None
            if dur is not None: i += 1
            loc = tokens[i] if i < n and is_location(tokens[i]) else None
            if loc is not None: i += 1

            entry = {
                'company_name': designation,
                'duration': org,
                'designations': [{
                    'designation': designation,
                    **({'duration': dur} if dur else {}),
                    **({'location': loc} if loc else {}),
                    'projects': ''
                }]
            }
            result.append(entry)

    return result

# --- main ---
tokens = get_experience_tokens(driver)
experience_struct = parse_experience_tokens(tokens)

profile_data["experience"] = experience_struct

# print(json.dumps(profile_data, indent=4, ensure_ascii=False))


In [ ]:
profile_data

Education 

In [ ]:
# --- Extract Education Section ---
edu_section = WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.XPATH, "//section[.//span[text()='Education']]"))
)

# Get HTML of education section only
edu_html = edu_section.get_attribute("outerHTML")
edu_soup = BeautifulSoup(edu_html, "lxml")

# Collect each education block (li elements usually represent entries)
edu_blocks = edu_soup.find_all("li", {"class": "artdeco-list__item"})

education_data = []

for block in edu_blocks:
    spans = block.find_all("span", {"class": "visually-hidden"})
    if not spans:
        continue

    edu_dict = {}
    edu_dict["college"]  = spans[0].get_text(strip=True) if len(spans) > 0 else ""
    edu_dict["degree"]   = spans[1].get_text(strip=True) if len(spans) > 1 else ""
    edu_dict["duration"] = spans[2].get_text(strip=True) if len(spans) > 2 else ""
    edu_dict["grade"]    = spans[3].get_text(strip=True) if len(spans) > 3 else ""

    education_data.append(edu_dict)

profile_data["education"] = education_data

# profile_data


In [ ]:
profile_data

Certificate

In [ ]:
import time

# open full license section
driver.find_element(By.ID, "navigation-index-see-all-licenses-and-certifications").click()
time.sleep(2)

# parse page
page_source = driver.page_source
soup = BeautifulSoup(page_source, 'lxml')

# find the section containing licenses
licenses_section = soup.find('section', {'class': 'artdeco-card pb3'})
if licenses_section:
    items = licenses_section.find_all('li', {'class': 'pvs-list__paged-list-item'})
else:
    items = []

def get_license(item):
    spans = item.find_all('span', {'class': 'visually-hidden'})
    item_dict = {}

    if len(spans) > 0:
        item_dict['name'] = spans[0].get_text().strip()
    if len(spans) > 1:
        item_dict['institute'] = spans[1].get_text().strip()
    if len(spans) > 2:
        item_dict['issued_date'] = spans[2].get_text().strip()

    return item_dict

item_list = []
for item in items:
    item_list.append(get_license(item))

profile_data['licenses'] = item_list
profile_data


In [ ]:
driver.back()

Project

In [ ]:
# import time
# from bs4 import BeautifulSoup
# from selenium.webdriver.common.by import By

# open full projects section
driver.find_element(By.ID, "navigation-index-see-all-projects").click()
time.sleep(2)   # wait for page to load

# parse page
page_source = driver.page_source
soup = BeautifulSoup(page_source, 'lxml')

# find projects section
projects_section = soup.find('section', {'class': 'artdeco-card pb3'})
if projects_section:
    items = projects_section.find_all('li', {'class': 'pvs-list__paged-list-item'})
else:
    items = []

def get_project(item): 
    spans = item.find_all('span', {'class': 'visually-hidden'})
    item_dict = {}

    if len(spans) > 0:
        item_dict['project_name'] = spans[0].get_text().strip()
    if len(spans) > 1:
        item_dict['duration'] = spans[1].get_text().strip()
    if len(spans) > 2:
        item_dict['description'] = spans[2].get_text().strip()
    if len(spans) > 3:
        item_dict['skills'] = spans[3].get_text().strip()

    return item_dict

# collect all projects
item_list = [get_project(item) for item in items]

# save in profile_data
profile_data['projects'] = item_list

# go back to main profile
# driver.back()
# time.sleep(2)

# profile_data


In [ ]:
profile_data


In [ ]:
driver.back()

Courses

In [ ]:
# Get page source after clicking "See all courses"
page_source = driver.page_source
soup = BeautifulSoup(page_source, 'lxml')

# Find all course items safely
items = soup.find_all('li', {'class': 'pvs-list__paged-list-item'})

courses_list = []
for item in items:
    spans = item.find_all('span', {'class': 'visually-hidden'})
    
    item_dict = {}
    item_dict['course_name'] = spans[0].get_text().strip() if len(spans) > 0 else ""
    item_dict['associated_with'] = spans[1].get_text().strip() if len(spans) > 1 else ""

    courses_list.append(item_dict)

# Add to your profile data
profile_data["courses"] = courses_list
# profile_data


In [ ]:
profile_data

MongoDB code

In [ ]:
import json
from pymongo import MongoClient

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["Linkedin"]
collection = db["profile_data"]

# Load data from JSON file
with open("profile_data.json", "r", encoding="utf-8") as file:
    data = json.load(file)

# If it's a single object
if isinstance(data, dict):
    collection.insert_one(data)
# If it's a list of objects
elif isinstance(data, list):
    collection.insert_many(data)

print("✅ Data added to MongoDB successfully!")


About section

In [ ]:
driver.find_element(By.CLASS_NAME,"inline-show-more-text__button").click()

In [ ]:
page_source = driver.page_source
soup = BeautifulSoup(page_source, 'lxml')

about = soup.find('div',{'class':'display-flex ph5 pv3'})
about = about.get_text().strip()

profile_data['about'] = about

In [ ]:
profile_data

About trial

In [15]:
driver.find_element(By.CLASS_NAME,"inline-show-more-text__button").click()

In [16]:
page_source = driver.page_source
soup = BeautifulSoup(page_source, 'lxml')

about_div = soup.find('div', {'class': 'display-flex ph5 pv3'})
if about_div:
    about = about_div.get_text().strip()
    profile_data['about'] = about if about else None
else:
    profile_data['about'] = None

# profile_data


In [17]:
profile_data

{'name': 'Dhruvi Lad',
 'url': 'https://www.linkedin.com/in/dhruvi-lad-7a900b352/',
 'about': 'To leverage and further develop my skills in software development, Internet of Things (IoT), and data analysis by contributing to impactful and innovative projects in the IT industry. I aim to work in a dynamic and challenging environment that fosters continuous learning, encourages creative problem-solving, and allow me to make meaningful contributions to technological advancements.To leverage and further develop my skills in software development, Internet of Things (IoT), and data analysis by contributing to impactful and innovative projects in the IT industry. I aim to work in a dynamic and challenging environment that fosters continuous learning, encourages creative problem-solving, and allow me to make meaningful contributions to technological advancements.'}

about-trial-1

In [29]:
from selenium.common.exceptions import NoSuchElementException
from bs4 import BeautifulSoup

# 1️⃣ Try clicking "See more" if it exists
try:
    driver.find_element(By.CLASS_NAME,"inline-show-more-text__button").click()
except NoSuchElementException:
    pass  # Button not present → ignore

# 2️⃣ Parse page
page_source = driver.page_source
soup = BeautifulSoup(page_source, 'lxml')

# 3️⃣ Look for About section
about_div = soup.find('div', {'class': 'display-flex ph5 pv3'})
if not about_div:  # fallback: sometimes it's in a <section>
    about_div = soup.find('section', {'id': 'about'})

if about_div:
    about = about_div.get_text().strip()
    profile_data['about'] = about if about else None
else:
    profile_data['about'] = None

# profile_data


In [30]:
profile_data

{'name': 'Mukund Savaliya',
 'url': 'https://www.linkedin.com/in/savaliya-mukund27/',
 'about': None}

experience no code

In [12]:
def clean_text(s: str) -> str:
    """Remove unwanted noise from tokens like hashtags, announcements, pdfs"""
    noise_patterns = ["hello,", "happy to share", "hashtag#", ".pdf", "joining intent", "new position"]
    s_lower = s.lower()
    if any(p in s_lower for p in noise_patterns):
        return ""
    return s.strip()

def parse_experience_tokens(tokens):
    if tokens and tokens[0].strip().lower() == "experience":
        tokens = tokens[1:]

    roles_set = {
        "Full-time","Part-time","Internship","Contract","Self-employed",
        "Freelance","Apprenticeship","Volunteer","Temporary","Seasonal"
    }

    def is_duration(s: str) -> bool:
        s2 = s.lower()
        return (' to ' in s2 and ('present' in s2 or 'mo' in s2 or 'yr' in s2)) or (' yr' in s2) or (' mo' in s2)

    def is_location(s: str) -> bool:
        if is_duration(s): return False
        if s in roles_set: return False
        loc_keywords = ['india','remote','on-site','hybrid','onsite','work from home']
        if any(k in s.lower() for k in loc_keywords):
            return True
        return (',' in s and len(s.split()) >= 2)

    def is_meta(s: str) -> bool:
        return s in roles_set or is_duration(s) or is_location(s) or s.lower() in ('on-site','remote','hybrid')

    i, n = 0, len(tokens)
    result = []

    while i < n:
        designation = clean_text(tokens[i]); i += 1
        if not designation:  # skip junk
            continue

        org = clean_text(tokens[i]) if i < n else None
        if org: i += 1

        dur = tokens[i] if i < n and is_duration(tokens[i]) else None
        if dur: i += 1

        loc = tokens[i] if i < n and is_location(tokens[i]) else None
        if loc: i += 1

        # 🔹 Only keep structured entries
        if designation and org and not any(k in org.lower() for k in ["hello", "hashtag", ".pdf"]):
            entry = {
                'company_name': org,
                'duration': dur,
                'designations': [{
                    'designation': designation,
                    **({'duration': dur} if dur else {}),
                    **({'location': loc} if loc else {}),
                    'projects': ''  # you can later extract project descriptions here
                }]
            }
            result.append(entry)

    return result
tokens = get_experience_tokens(driver)
experience_struct = parse_experience_tokens(tokens)

profile_data["experience"] = experience_struct


NameError: name 'get_experience_tokens' is not defined